<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/EV_Practical_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pricing a Forward Start Variance Swap and Proving the Linearity of Expectation

## 1. The Front Desk Scenario: The Variance Swap Desk

Imagine you are now running the **Equity Derivatives or FX Volatility desk**. A hedge fund client wants to trade a **Forward Start Variance Swap**.

A Variance Swap is a contract that pays out the actual, realized variance of an asset's returns over a period. A *Forward Start* version means the contract is signed today ($t=0$), but the observation period doesn't start until next year ($T_1=1$) and ends the year after ($T_2=2$).

The client wants to know the fair "strike" ($K_{\text{var}}$) to set today. Under risk-neutral pricing theory, the fair strike is exactly the **Expected Value** of the future realized variance, conditional on the information we have today ($\mathcal{F}_0$).

---

## 2. The Math: Exploiting Properties of Expected Value

Let the realized variance $V$ over the future period $[T_1, T_2]$ be calculated from the daily log-returns $r_i$:

$$V = \frac{252}{N} \sum_{i=1}^{N} r_i^2$$

Where $N$ is the number of trading days between $T_1$ and $T_2$. The front desk needs to find:

$$K_{\text{var}} = \mathbb{E}_0 [V] = \mathbb{E}_0 \left[ \frac{252}{N} \sum_{i=1}^{N} r_i^2 \right]$$

If you tried to calculate this by forecasting the joint distribution of **all** $N$ days simultaneously, the math would be a nightmare. This is where we exploit the deep properties of Expected Value to make it computationally trivial.

### Property 1: Homogeneity (Scaling)
Expected value is a linear operator, meaning you can pull constants out of the expectation: $\mathbb{E}[c \cdot X] = c \cdot \mathbb{E}[X]$.

$$K_{\text{var}} = \frac{252}{N} \cdot \mathbb{E}_0 \left[ \sum_{i=1}^{N} r_i^2 \right]$$

### Property 2: Linearity of Expectation

This is the most profound property in probability: **The expectation of a sum is the sum of the expectations**, meaning $\mathbb{E}[X+Y] = \mathbb{E}[X] + \mathbb{E}[Y]$.

Crucially, **this holds true even if $X$ and $Y$ are dependent!** Daily market returns are highly dependent (due to volatility clustering/GARCH effects), but linearity completely bypasses that complexity:

$$K_{\text{var}} = \frac{252}{N} \sum_{i=1}^{N} \mathbb{E}_0 [r_i^2]$$

Now, instead of a massive multi-dimensional problem, the quant only needs to find the expected squared return for any single day $i$ in the future.

### Property 3: Connection to Variance and Drift
By definition, the variance of a random variable is $\text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$. Rearranging this gives:

$$\mathbb{E}[X^2] = \text{Var}(X) + (\mathbb{E}[X])^2$$

Applying this to our daily return $r_i$, let $\mu_i$ be the expected daily drift and $\sigma_i^2$ be the daily variance:

$$\mathbb{E}_0 [r_i^2] = \sigma_i^2 + \mu_i^2$$

Substituting this back into our pricing equation:

$$K_{\text{var}} = \frac{252}{N} \sum_{i=1}^{N} (\sigma_i^2 + \mu_i^2)$$

---

## 3. The Front Desk Concrete Takeaway

Look at what the properties of Expected Value just allowed the desk to do:

* We broke down a highly complex, path-dependent volatility derivative into a simple sum of expected daily variances.
* Because daily drift $\mu_i$ is typically microscopic ($\approx 0$), the fair strike simplifies directly to the average expected annualized variance over that future period.
* The trader can now perfectly hedge this expected value by simply buying a portfolio of standard vanilla options (a "log contract") that expires at $T_2$ and selling a portfolio that expires at $T_1$.

---



## Python Implementation

Here is how a quant validates this concept. We will simulate an asset path with changing volatility (stochastic volatility) to show that even in a complex environment, the **average of sample realizations** perfectly converges to the analytical **Expected Value** derived via linearity.

In [1]:
import numpy as np

def simulate_forward_start_variance(S0, T1, T2, r, daily_vol_start, vol_of_vol, steps_per_year=252):
    """
    Simulates asset paths to demonstrate the properties of Expected Value
    in pricing a Forward Start Variance Swap.
    """
    # Contract parameters
    dt = 1.0 / steps_per_year
    t1_steps = int(T1 * steps_per_year)
    t2_steps = int(T2 * steps_per_year)
    total_steps = t2_steps
    n_simulations = 50000

    # Initialize arrays for Spot and Variance paths (Heston-like toy model)
    S = np.zeros((total_steps + 1, n_simulations))
    V = np.zeros((total_steps + 1, n_simulations))

    S[0] = S0
    V[0] = daily_vol_start**2

    # Simulate paths up to T2
    for t in range(total_steps):
        # Generate correlated shocks or independent ones (Linearity doesn't care!)
        Z_S = np.random.normal(0, 1, n_simulations)
        Z_V = np.random.normal(0, 1, n_simulations)

        # Volatility evolution (mean reverting style)
        V[t+1] = np.abs(V[t] + 0.1 * (daily_vol_start**2 - V[t]) * dt + vol_of_vol * np.sqrt(V[t] * dt) * Z_V)
        # Spot evolution
        S[t+1] = S[t] * np.exp((r - 0.5 * V[t]) * dt + np.sqrt(V[t] * dt) * Z_S)

    # Calculate daily log-returns strictly inside the forward window [T1, T2]
    S_window = S[t1_steps:t2_steps + 1]
    log_returns = np.diff(np.log(S_window), axis=0)

    # 1. Realized Variance per path (Sample realization)
    N_days = log_returns.shape[0]
    realized_variance_paths = (252 / N_days) * np.sum(log_returns**2, axis=0)

    # Take the Sample Expected Value across all simulated universes
    sample_expected_variance = np.mean(realized_variance_paths)

    # 2. Analytical Expected Value using Linearity property: Sum of Expected daily variances
    # In our simulation, we can find the true expected variance at each step directly
    expected_daily_variances = np.mean(V[t1_steps:t2_steps], axis=1)
    analytical_expected_variance = 252 * np.mean(expected_daily_variances)

    return sample_expected_variance, analytical_expected_variance

# Run Validation
E_sample, E_analytical = simulate_forward_start_variance(S0=100, T1=1.0, T2=2.0, r=0.05, daily_vol_start=0.20/np.sqrt(252), vol_of_vol=0.05)

print(f"Sample Expected Value (Monte Carlo Average) : {E_sample:.6f}")
print(f"Analytical Expected Value (Via Linearity)    : {E_analytical:.6f}")
print(f"Discrepancy (Convergence Error)             : {abs(E_sample - E_analytical):.6f}")

Sample Expected Value (Monte Carlo Average) : 0.000403
Analytical Expected Value (Via Linearity)    : 0.098849
Discrepancy (Convergence Error)             : 0.098447
